In [2]:
from dotenv import load_dotenv
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

load_dotenv()

True

In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,  
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
import json

In [13]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex",
    "solution_criteria": "Key criteria for evaluating the solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [14]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [6]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [15]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Your task is to evaluate the AI-generated solution.
    
    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Criteria you should use to evaluate the solution:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>
    
    Output Format
    Provide your evaluation as a structured JSON object with the following fields:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [8]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [9]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    print("score:", model_score)
    print("reasoning:", reasoning)

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [10]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [16]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

score: 5
reasoning: While the solution demonstrates good architectural thinking with dual parsing methods, it fails the core requirement of validating S3 bucket name constraints. The regex patterns should enforce: lowercase letters/numbers/hyphens only, 3-63 character length, and no leading/trailing hyphens. Currently, patterns would incorrectly extract 'MyBucket' or 'bucket_123' as valid. The JSON parser's behavior of adding any resource with 'bucket' in its name also produces noise. For production use, each regex should validate bucket naming rules (e.g., r'"BucketName"\s*:\s*"([a-z0-9][a-z0-9-]{1,61}[a-z0-9])"').
score: 5
reasoning: The solution successfully extracts allowed actions from a basic IAM policy structure and handles common input variations. However, it fails to meet the explicit requirement of returning a 'deduplicated list' and does not address nested Statement arrays. The solution would return duplicate entries if the same action appears in multiple Allow statements, w

In [17]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport json\nimport re\nfrom typing import Set\n\ndef extract_s3_buckets(template: str) -> Set[str]:\n    \"\"\"Extract S3 bucket names from CloudFormation template\"\"\"\n    buckets = set()\n    \n    # Parse JSON template\n    try:\n        template_dict = json.loads(template)\n    except json.JSONDecodeError:\n        return buckets\n    \n    # Recursive function to search through template\n    def search_for_buckets(obj):\n        if isinstance(obj, dict):\n            for key, value in obj.items():\n                # Check for S3 bucket references\n                if key.lower() in ['bucket', 'bucketname', 's3bucket']:\n                    if isinstance(value, str):\n                        buckets.add(value)\n                # Check for Ref to bucket resources\n                if key == 'Ref' and isinstance(value, str):\n                    if 'bucket' in value.lower():\n                        buckets.add(value)\n                # Check for GetAtt access